# Bonus Challenge Five: Deploying Agents

**Goal:** Demonstrate the ability to deploy and use an agent using Google Agent Platform
(Vertex AI Agent Engine).

**Requirements covered in this notebook:**
1. Create an agent using the ADK -- this reuses the search -> critique -> refine
   workflow (`greeter` / `answer_team`) built in Challenge Four.
2. Deploy the agent to Agent Platform (`agent_engines.create(...)`).
3. Test the deployed agent.




In [1]:
# 1. Install dependencies
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.9/233.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.7 MB/s eta 0:00:00


In [2]:
# 2. Imports and configuration
import os
import logging
from typing import Optional

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

# --- Configuration ---
# GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION: your Cloud Skills Boost lab project ID
#   (shown on your Qwiklabs lab page) and a Vertex AI region.
# STAGING_BUCKET: a Cloud Storage bucket (gs://...) Agent Engine uses to stage the
#   deployment package. Create one first if you don't have one, e.g.:
#     gsutil mb -l us-central1 gs://YOUR_PROJECT_ID-agent-engine-staging

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-00-6263dfcac21a")
GOOGLE_CLOUD_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
STAGING_BUCKET = os.environ.get("STAGING_BUCKET", "gs://agent_search_adk")

import vertexai
vertexai.init(
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent_deployment")


## The agent (same workflow as Challenge Four)

`search_agent` -> `critique_agent` -> `refine_agent`, chained in `answer_team`, with
`greeter` as the root entry point.


In [3]:
# 3. Search agent: researches the question and drafts an initial answer
SEARCH_AGENT_INSTRUCTIONS = """
You are a research assistant. Use Google Search to find accurate, up-to-date
information that answers the user's question.

Write a clear, well-supported draft answer based on what you find. Keep it
factual and cite specifics (names, numbers, dates) where relevant. This is a
first draft -- a reviewer will critique it next, so it does not need to be
perfect, but it should be accurate and directly address the question.
"""

search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI_FLASH,
    description="Researches the user's question with Google Search and writes a draft answer.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    output_key="draft_answer",
    # The built-in google_search tool cannot be combined with any other
    # function-declaration tool (e.g. an auto-injected transfer tool) in the
    # same model call. This agent has no siblings/parent that need it to
    # transfer control, so disable that mechanism defensively.
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)


In [4]:
# 4. Critique agent: reviews the draft and suggests improvements
CRITIQUE_AGENT_INSTRUCTIONS = """
You are a careful editorial reviewer. You will be shown a draft answer to a
user's question:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

Review it for accuracy, completeness, and clarity. Write a short, specific,
actionable list of suggestions for how to improve it (e.g. missing details,
unclear phrasing, unsupported claims). If the draft is already excellent and
needs no changes, say so explicitly and clearly (e.g. "No changes needed.").

Only output the review notes -- do not rewrite the answer yourself.
"""

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_GEMINI_FLASH,
    description="Reviews the draft answer and suggests concrete improvements.",
    instruction=CRITIQUE_AGENT_INSTRUCTIONS,
    output_key="critique_notes",
)


In [5]:
# 5. Refine agent: rewrites the draft using the critique
REFINE_AGENT_INSTRUCTIONS = """
You will be shown a draft answer and a reviewer's critique of it:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

--- REVIEWER NOTES ---
{critique_notes}
--- END REVIEWER NOTES ---

Rewrite the draft answer, applying the reviewer's suggestions (if the notes
say no changes are needed, just clean up the draft's wording). Output only
the final, polished answer to the user's original question -- no
meta-commentary about the review process.
"""

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_GEMINI_FLASH,
    description="Rewrites the draft answer to incorporate the reviewer's suggested improvements.",
    instruction=REFINE_AGENT_INSTRUCTIONS,
    output_key="final_answer",
)


In [6]:
# 6. Sequential workflow and root agent
answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Answers a question by researching a draft, critiquing it, and "
        "refining it into a final response."
    ),
    sub_agents=[search_agent, critique_agent, refine_agent],
)

GREETER_INSTRUCTIONS = """
You are the friendly entry point for a question-answering assistant. You do
not answer questions yourself. As soon as the user asks a question, delegate
it to the `answer_team` sub-agent, which will research, critique, and refine
a high-quality response before it is shown to the user.
"""

greeter = Agent(
    name="greeter",
    model=MODEL_GEMINI_FLASH,
    description="Entry point that greets the user and delegates their question to the answer team.",
    instruction=GREETER_INSTRUCTIONS,
    sub_agents=[answer_team],
)


/tmp/ipykernel_34907/429872718.py:2: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


## Step 3: test the agent locally first

Per the workshop's deployment steps, test locally with `AdkApp` *before* deploying


In [11]:
# 7. Local helper to run a query and print every event
from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display

def describe_event(event: dict) -> None:
    """
    Print a one-line-per-part summary of a single streamed event: which agent
    authored it, and whether it is text, a tool call, or a tool result.

    Args:
        event (dict): One event dict from an AdkApp/AgentEngine stream_query().
    """
    author = event.get("author", "?")
    content = event.get("content") or {}
    for part in content.get("parts", []) or []:
        if part.get("text"):
            text = part.get("text", "").strip()[:200]
            print(f"  [{author}] TEXT  » {text}")
        elif part.get("function_call"):
            fc = part["function_call"]
            fc_name = fc.get("name")
            fc_args = fc.get("args")
            print(f"  [{author}] CALL  » {fc_name}({fc_args})")
        elif part.get("function_response"):
            fr = part["function_response"]
            fr_name = fr.get("name")
            print(f"  [{author}] RESULT » from {fr_name}")


def ask_agent_verbose(app, question: str, user_id: str = "test-user-id") -> Optional[str]:
    """
    Create a session on the given app/agent, query it once, and print every
    event along the way before returning the final response text. Works for
    both a local AdkApp and a deployed remote AgentEngine, since both expose
    the same create_session/stream_query interface.

    Args:
        app: A local `AdkApp` or a deployed `AgentEngine` (from agent_engines.create()).
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        Optional[str]: The text of the final response, or None on error.
    """
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    event_count = 0
    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=question):
            describe_event(event)
            last_event = event
            event_count += 1
    except Exception as e:
        print(f"Error while querying agent: {e}")
        return None

    if not last_event or "content" not in last_event:
        print(f"Agent did not return a valid final response ({event_count} event(s) received).")
        # Print the raw last event so we can see exactly what came back --
        # e.g. an error_code/error_message, or an empty actions payload --
        # instead of guessing why the stream ended early.
        print("Raw last event:", last_event)
        return None

    return last_event["content"]["parts"][0]["text"]

In [8]:
# 8. Test locally before deploying
local_app = AdkApp(agent=greeter)

question = "What is the Google Agent Development Kit (ADK) and what is it used for?"
print(f"=== Local test: {question} ===")
local_response = ask_agent_verbose(local_app, question)
print()
display(Markdown(local_response or "*(no response)*"))


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


=== Local test: What is the Google Agent Development Kit (ADK) and what is it used for? ===


/usr/local/lib/python3.12/dist-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()


  [greeter] CALL  » transfer_to_agent({'agent_name': 'answer_team'})
  [greeter] RESULT » from transfer_to_agent
  [search_agent] TEXT  » The Google Agent Development Kit (ADK) is an open-source framework developed by Google designed to help developers build, evaluate, and deploy sophisticated AI agents and multi-agent systems. It focus
  [critique_agent] TEXT  » Here are a few suggestions to improve the draft answer:

1.  **Refine "Enterprise-Scale AI Agents" point:** This point largely repeats aspects mentioned in "Creating AI-Powered Applications" (producti
  [refine_agent] TEXT  » The Google Agent Development Kit (ADK) is an open-source framework developed by Google designed to help developers build, evaluate, and deploy sophisticated AI agents and multi-agent systems. It focus



The Google Agent Development Kit (ADK) is an open-source framework developed by Google designed to help developers build, evaluate, and deploy sophisticated AI agents and multi-agent systems. It focuses on treating agents as software components rather than simple prompt-based workflows, enabling more robust and scalable AI applications.

The ADK is used for:
*   **Building Smart AI Agents** The primary purpose of ADK is to facilitate the creation of intelligent AI agents capable of reasoning, remembering information, and interacting with one another.
*   **Developing Multi-Agent Systems** ADK is built for a world where agents collaborate. It enables developers to create systems with multiple specialized agents that work together as a team to solve complex problems, rather than relying on a single, monolithic agent. This multi-agent architecture allows for modular and scalable applications with complex coordination and delegation.
*   **Building Production-Ready and Enterprise-Scale AI Applications** ADK provides a robust, event-driven framework for developing AI applications that move beyond simple chat interfaces. It enables the creation, debugging, and deployment of reliable and scalable agentic applications, offering the flexibility and precise control needed for enterprise-grade solutions.
*   **Evaluating and Debugging Agents** The framework supports the assessment and refinement of agent behavior. Through its event-driven architecture, ADK provides observability and debugging capabilities essential for testing, validating, and improving agents' performance and reliability before and after deployment.
*   **Autonomous Workflows** The framework simplifies the creation of multi-tool, autonomous workflows, allowing agents to plan, reason, and execute tasks.

Key features of Google ADK include:
*   **Event-driven Architecture** It operates as a sophisticated event loop that mediates between user requests, AI model invocations, and external tool executions, providing observability and debugging capabilities crucial for evaluation.
*   **Integrated Memory and Goal Tracking** ADK uses "Artifacts"—structured representations of information—for integrated memory and goal tracking, helping agents maintain state and context across interactions.
*   **Multimodal Data Support** It natively supports multimodal data types, including documents, audio, and video.
*   **Google Agent Protocol** The ADK includes built-in support for Google's Agent Protocol, which enables inter-agent communication.
*   **Integration with Google Ecosystem** While flexible enough to work with any AI model, ADK is deeply integrated with Google's ecosystem, including Gemini and Vertex AI, and optimized for seamless integration within Google Cloud.
*   **Code-First Approach** ADK applies software development principles to AI agent creation, making it a code-first framework where agents are defined as objects and tools are regular functions.

## Step 4: deploy the agent to Agent Platform

`agent_engines.create(...)` packages the app and its dependencies, uploads them to the
staging bucket, and provisions a managed, scalable endpoint for it on Vertex AI Agent
Engine.


In [9]:
# 9. Deploy to Agent Platform (Vertex AI Agent Engine)
from vertexai import agent_engines

remote_agent = agent_engines.create(
    local_app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
    display_name="adk-workshop-answer-team",
    description="Search -> critique -> refine question-answering workflow (Challenge 4/5).",
)

print("Deployed resource name:", remote_agent.resource_name)


INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.13.4', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.165.1'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket agent_search_adk
INFO:vertexai.agent_engines:Wrote to gs://agent_search_adk/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://agent_search_adk/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://agent_search_adk/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/386462097804/locations/us-central1/reasoningEngines/630905654405

Deployed resource name: projects/386462097804/locations/us-central1/reasoningEngines/6309056544050774016


## Test the deployed agent





In [12]:
# 10. Test the deployed (remote) agent
remote_question = "What is the capital of France, and what is it known for?"
print(f"=== Remote test: {remote_question} ===")
remote_response = ask_agent_verbose(remote_agent, remote_question)
print()
display(Markdown(remote_response or "*(no response)*"))


=== Remote test: What is the capital of France, and what is it known for? ===
  [greeter] CALL  » transfer_to_agent({'agent_name': 'answer_team'})
  [greeter] RESULT » from transfer_to_agent
  [search_agent] TEXT  » Paris is the capital and largest city of France.

It is renowned globally for numerous aspects, earning it the nickname "City of Light" due to its historical role in the Age of Enlightenment and its e
  [critique_agent] TEXT  » No changes needed.
  [refine_agent] TEXT  » Paris is the capital and largest city of France.

It is renowned globally for numerous aspects, earning it the nickname "City of Light" due to its historical role in the Age of Enlightenment and its e



Paris is the capital and largest city of France.

It is renowned globally for numerous aspects, earning it the nickname "City of Light" due to its historical role in the Age of Enlightenment and its early adoption of gas street lighting. Paris is a major hub for finance, diplomacy, commerce, culture, fashion, and gastronomy.

The city is recognized for its iconic landmarks, including the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. It houses world-famous art collections, such as those at the Musée d'Orsay, Musée Marmottan Monet, and the Louvre, which is home to the Mona Lisa. Paris has also been a crucible for various artistic movements, including Impressionism and Art Deco.

Furthermore, Paris is considered a world capital of shopping and fashion, hosting prominent brands like Chanel, Dior, and Louis Vuitton. Its unique urban landscape, characterized by grand boulevards and Haussmannian architecture, contributes to its charm. The city's rich history, vibrant culture, and esteemed academic institutions also make it a significant global destination, attracting millions of tourists each year.